In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)

from src.utils.features import split_features
from src.utils.notebook_ploting import correlation_heatmap

DATA_DIR = Path("../datasets/final/ml")

DATASET_NAMES = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
]

THRESHOLD = 0.85

In [2]:
# limiar para comparação. threshold => 0.85 -> alta correlação
# definido no pré-tcc
THRESHOLD = 0.85

# correlação e correlação negativa tem o mesmo impacto
all_high_corr_pairs = []

for dataset_name in DATASET_NAMES:
    print(f"\n{dataset_name}")

    df = pd.read_parquet(
        DATA_DIR / f"{dataset_name}_train.parquet"
    )

    numerical_features, _ = split_features(df)
    corr_matrix = df[numerical_features].corr(method="pearson")

    correlation_heatmap(corr_matrix, is_abs=True)

    corr_abs = corr_matrix.abs()

    # pega apenas parte de cima da matriz, tambémn ignora a diagonal. evita cálculo repetido
    #ex:
    # 1 2 3 -> nan  2   3
    # 4 5 6 -> nan nan  3
    # 7 8 9 -> nan nan nan

    # np.ones(corr_abs.shape) -> cria matriz de 1 com o shape
    # np.triu( -> pega apenas triângulo superior
    # k = 1 exclui diagonal
    # transforma em boleano
    # em resumo, uma máscara boleana para a matriz

    upper = corr_abs.where(
        np.triu(np.ones(corr_abs.shape), k=1).astype(bool)
    )

    # upper.stack() -> empilha as colunas para transformar em linhas, já filtra nan
    # .reset_index() -> transforma series em tabela
    # renomear os nomes de coluna que o pandas atribui
    #display(upper.stack())
    corr_pairs = (
        upper.stack()
        .reset_index()
        .rename(columns={
            "level_0": "feature_1",
            "level_1": "feature_2",
            0: "abs_correlation",
        })
    )

    high_corr_pairs = corr_pairs[
        corr_pairs["abs_correlation"] >= THRESHOLD
    ].sort_values(by="abs_correlation", ascending=False)

    high_corr_pairs.insert(0, "dataset", dataset_name)

    display(high_corr_pairs)

    all_high_corr_pairs.append(high_corr_pairs)

    del df

all_high_corr_pairs = pd.concat(
    all_high_corr_pairs,
    ignore_index=True,
)


cicids2017


,dataset,feature_1,feature_2,abs_correlation
28,cicids2017,bidirectional_packets,bidirectional_ack_packets,0.999999
18,cicids2017,bidirectional_packets,bidirectional_bytes,0.999908
44,cicids2017,bidirectional_bytes,bidirectional_ack_packets,0.999908
86,cicids2017,bidirectional_stddev_ps,bidirectional_max_ps,0.990174
69,cicids2017,bidirectional_mean_ps,bidirectional_stddev_ps,0.942631
205,cicids2017,bidirectional_ack_packets,bidirectional_psh_packets,0.938712
29,cicids2017,bidirectional_packets,bidirectional_psh_packets,0.938700
45,cicids2017,bidirectional_bytes,bidirectional_psh_packets,0.937242
137,cicids2017,bidirectional_mean_piat_ms,bidirectional_stddev_piat_ms,0.926994
154,cicids2017,bidirectional_stddev_piat_ms,bidirectional_max_piat_ms,0.926695



unsw_nb15


,dataset,feature_1,feature_2,abs_correlation
28,unsw_nb15,bidirectional_packets,bidirectional_ack_packets,0.999641
191,unsw_nb15,bidirectional_syn_packets,bidirectional_fin_packets,0.973032
18,unsw_nb15,bidirectional_packets,bidirectional_bytes,0.969131
44,unsw_nb15,bidirectional_bytes,bidirectional_ack_packets,0.968862
137,unsw_nb15,bidirectional_mean_piat_ms,bidirectional_stddev_piat_ms,0.940051
86,unsw_nb15,bidirectional_stddev_ps,bidirectional_max_ps,0.931930



iot23


,dataset,feature_1,feature_2,abs_correlation
239,iot23,bidirectional_rst_packets,bidirectional_fin_packets,0.998399
154,iot23,bidirectional_stddev_piat_ms,bidirectional_max_piat_ms,0.989231
138,iot23,bidirectional_mean_piat_ms,bidirectional_max_piat_ms,0.980375
205,iot23,bidirectional_ack_packets,bidirectional_psh_packets,0.979948
44,iot23,bidirectional_bytes,bidirectional_ack_packets,0.971362
52,iot23,bidirectional_min_ps,bidirectional_mean_ps,0.968235
28,iot23,bidirectional_packets,bidirectional_ack_packets,0.963812
137,iot23,bidirectional_mean_piat_ms,bidirectional_stddev_piat_ms,0.961839
18,iot23,bidirectional_packets,bidirectional_bytes,0.952155
120,iot23,bidirectional_min_piat_ms,bidirectional_mean_piat_ms,0.946689


In [3]:
common_pairs = (
    all_high_corr_pairs
    .pivot(
        index=["feature_1", "feature_2"],
        columns="dataset",
        values="abs_correlation",
    )
    .dropna()
)

display(common_pairs)

dataset                                                  cicids2017     iot23  unsw_nb15
feature_1                  feature_2                                                    
bidirectional_bytes        bidirectional_ack_packets       0.999908  0.971362   0.968862
bidirectional_mean_piat_ms bidirectional_stddev_piat_ms    0.926994  0.961839   0.940051
bidirectional_packets      bidirectional_ack_packets       0.999999  0.963812   0.999641
                           bidirectional_bytes             0.999908  0.952155   0.969131